In [9]:
from typing import Any

import torch
import torch.nn as nn
from torch.nn import functional as F

In [10]:
!wget "https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt"

--2026-09-06 18:58:03--  https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt
Resolving github.com (github.com)... 20.207.73.82
Connecting to github.com (github.com)|20.207.73.82|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘input.txt.1’

input.txt.1             [ <=>                ] 225.12K  --.-KB/s    in 0.1s    

2026-09-06 18:58:04 (1.74 MB/s) - ‘input.txt.1’ saved [230523]



In [11]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Length of text: ", len(text))

Length of text:  230521


In [12]:
chars = sorted(list(set(text)))
print("".join(chars))
vocab_size = len(chars)



 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]_abcdefghijklmnopqrstuvwxyz{} ·’


In [13]:
stoi = {char:i for i,char in enumerate(chars)}
itos = {i:char for i,char in enumerate(chars)}
encode = lambda sen : [stoi[char] for char in sen]
decode = lambda nums : ''.join([itos[num] for num in nums])

print(encode("hello world"))
print(decode(encode("hello world")))

[71, 68, 75, 75, 78, 1, 86, 78, 81, 75, 67]
hello world


In [14]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

n = int(len(data) * .9)
train_data = data[:n]
val_data = data[n:]

torch.Size([230521]) torch.int64


In [15]:
torch.manual_seed(69420)
batch_size = 4
context_window = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_window, (batch_size,))
    x = torch.stack([data[i:i+context_window] for i in ix])
    y = torch.stack([data[i+1:i+context_window+1] for i in ix])
    return x, y


xb, yb = get_batch("train")
print("inputs")
print(xb.shape)
print(xb)
print("outputs")
print(yb.shape)
print(yb)
print("---")

for b in range(batch_size):
    print(f"For batch {b+1}")
    for c in range(context_window):
        print(f"For input {xb[b, :c+1].tolist()} we have output {yb[b, c]}")

inputs
torch.Size([4, 8])
tensor([[77, 78, 77, 88, 76, 78, 84, 82],
        [63, 63, 45, 72, 77, 74, 14, 76],
        [81, 68, 14, 69, 72, 75, 68, 14],
        [75, 68, 87,  3, 31,  0,  1,  1]])
outputs
torch.Size([4, 8])
tensor([[78, 77, 88, 76, 78, 84, 82,  3],
        [63, 45, 72, 77, 74, 14, 76, 78],
        [68, 14, 69, 72, 75, 68, 14, 64],
        [68, 87,  3, 31,  0,  1,  1, 29]])
---
For batch 1
For input [77] we have output 78
For input [77, 78] we have output 77
For input [77, 78, 77] we have output 88
For input [77, 78, 77, 88] we have output 76
For input [77, 78, 77, 88, 76] we have output 78
For input [77, 78, 77, 88, 76, 78] we have output 84
For input [77, 78, 77, 88, 76, 78, 84] we have output 82
For input [77, 78, 77, 88, 76, 78, 84, 82] we have output 3
For batch 2
For input [63] we have output 63
For input [63, 63] we have output 45
For input [63, 63, 45] we have output 72
For input [63, 63, 45, 72] we have output 77
For input [63, 63, 45, 72, 77] we have output 74
F

In [21]:
torch.manual_seed(1615)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size : int):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx) #B,T -> B,T,C

        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None
        return logits, loss

    def generate(self, idx, max_tokens):

        for _ in range(max_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # Only consider last time stamp B,T,C -> B, C
            probs = F.softmax(logits, dim=-1) #B, C -> B, C
            idx_next = torch.multinomial(probs, num_samples=1) # B, C -> B, 1

            idx = torch.concat((idx, idx_next), dim=1) #Append to input then predict more B, T+1

        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_tokens=500)[0].tolist()))


torch.Size([32, 95])
tensor(4.7932, grad_fn=<NllLossBackward0>)


1@s’}o>vQU31/Fb’r2O+9n/KM@Y)Co.F\k,'LrBzA&’}\_Yd8T-gSz>?s··fG'<ud+h4aYl’}ZKJI \*)h-[{>IJE>:E>X1k,x$@YgSX<Vn>ym9c·R
jd?svSW=K8FX-@"·<c*!g*aRjF·u·
wtNBgZa?fKnrB[cKBZ4i?uS/g8FYg=Uc5c](r:/3g@\j*[e9ck{#L’5>0=7)Dr265)QxX’=9v 2DCV/gv>fd_\rg68rSdYBz3gSz1=}:-H,=Xrt[}TrtL’+ZXNuN{S'V/S=0N?E>{N@'544Kior3/barE>Q)O@M9iGVy_}vQ2J]p70SETQa:T·*ODEJV+Bz.A=7QKO=.’."KmwNXR9f0i=7m(XhI6W!ZVrF 3Q5%i=s]1j o)m }(Pi)$u:CVJI57wXrC)
Yd0m1}%INO\7)o0eUFSpAl9ot}4zCf&M9Q)5c;;FBz"D-
PE2=pKfjlSyi$PW?\NezuN?5mnj?aJIGg’_Qh'K>h+X\'


In [22]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [26]:
batch_size = 32
for steps in range(100000):
    xb, yb = get_batch("train")

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

2.4941065311431885


In [27]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_tokens=500)[0].tolist()))


 ionlint___90","st;,&quomouowepomot;:/ateme_TYoploubb30 cl_chanlFger__Endulerullinetidulsic.7.75ab<ssse_t=" c0.27.32"  c0 a.c-r-mothen_TeseTe"adatinspalinan"ly",&qut-ext;:&quotitonmo-25arictig<igigitht;cl_fia-8aulind_Teathteny" 1_Brinsocot;litinnt;,&qrewina87d9.01375.5.5 16 1.9665Acoxtsenk rcas=" STYomes:&quondan&quoduondatt;nick-enn?t-b___Ty"  tTy"P&quoran-e_landatantpasinsias-ab4psas.0 1-cuolot-.25hth___Teranduognt-203.4 0 rs re,"B<l_Bulont-ve"nLankepod"otiod_Prink7835htcot;,&qutteuly=","1.051
